# Computer Vision Using Deep Learning - Practical 8

## Transfer Learning

---

## 1. Setup and Base Model Definition

First, let's import the necessary libraries and define the base CNN architecture. We will assume we have a pre-trained model saved as `cifar_cnn.pth`. This model was originally trained on the CIFAR-10 dataset.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import warnings
from torchsummary import summary

warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        # Feature extractor layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        
        # Classifier layers
        self.fc1 = nn.Linear(64 * 8 * 8, 128) # Input images are 32x32, after two 2x2 pooling layers, the size becomes 8x8
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNN(num_classes=10).to(device)

# The input size must match what the network expects: (channels, height, width)
summary(model, (3, 32, 32))

Using device: cpu
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]             896
         MaxPool2d-2           [-1, 32, 16, 16]               0
            Conv2d-3           [-1, 64, 16, 16]          18,496
         MaxPool2d-4             [-1, 64, 8, 8]               0
            Linear-5                  [-1, 128]         524,416
            Linear-6                   [-1, 10]           1,290
Total params: 545,098
Trainable params: 545,098
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.01
Forward/backward pass size (MB): 0.47
Params size (MB): 2.08
Estimated Total Size (MB): 2.56
----------------------------------------------------------------


### Create and Save a Dummy Pre-trained Model
For this notebook to run without errors, we need the `cifar_cnn.pth` file. The following cell creates a dummy model with random weights and saves it. In a real-world scenario, this file would contain weights from a model fully trained on CIFAR-10.

In [ ]:

dummy_model = CNN(num_classes=10)
torch.save(dummy_model.state_dict(), "cifar_cnn.pth")

print("Dummy 'cifar_cnn.pth' created.")

Dummy 'cifar_cnn.pth' created.


---

## 2. Experiment 1: Transfer Learning on SVHN Dataset

The **Street View House Numbers (SVHN)** dataset contains images of digits and is similar in complexity and format to CIFAR-10 (32x32 color images). This makes it a great candidate for transfer learning.

First, we load the data.

In [8]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download and load the SVHN dataset
svhn_train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
svhn_train_loader = DataLoader(svhn_train_dataset, batch_size=64, shuffle=True)

svhn_test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)
svhn_test_loader = DataLoader(svhn_test_dataset, batch_size=64, shuffle=False)

Using downloaded and verified file: ./data\train_32x32.mat
Using downloaded and verified file: ./data\test_32x32.mat


### 2.1. Strategy A: Feature Extraction (Frozen Layers)

Here, we'll load the pre-trained weights, freeze the convolutional layers (`conv1`, `conv2`) and the first fully connected layer (`fc1`), and only train the final classification layer (`fc2`).

In [9]:

pretrained_model = CNN(num_classes=10)
pretrained_model.load_state_dict(torch.load("cifar_cnn.pth"))

#  new model for SVHN
svhn_model_frozen = CNN(num_classes=10)

# Copy weights from the pre-trained model, except for the final layer
pretrained_dict = pretrained_model.state_dict()
model_dict = svhn_model_frozen.state_dict()

# Filter out the final layer (fc2) from the pre-trained dictionary
pretrained_dict = {k: v for k, v in pretrained_dict.items() if "fc2" not in k}
model_dict.update(pretrained_dict)
svhn_model_frozen.load_state_dict(model_dict)

# Freeze all layers except the final one
for name, param in svhn_model_frozen.named_parameters():
    if "fc2" not in name:
        param.requires_grad = False

print("Frozen Model Parameter Status:")
for name, param in svhn_model_frozen.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

svhn_model_frozen.to(device)

# The optimizer will only update the parameters of the unfrozen layer (fc2)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, svhn_model_frozen.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss()

Frozen Model Parameter Status:
conv1.weight: requires_grad=False
conv1.bias: requires_grad=False
conv2.weight: requires_grad=False
conv2.bias: requires_grad=False
fc1.weight: requires_grad=False
fc1.bias: requires_grad=False
fc2.weight: requires_grad=True
fc2.bias: requires_grad=True


In [10]:
# Training loop for the frozen model
print("\n--- Training Frozen Model on SVHN ---")
num_epochs = 5
for epoch in range(num_epochs):
    svhn_model_frozen.train()
    running_loss = 0.0
    for images, labels in svhn_train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = svhn_model_frozen(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(svhn_train_loader):.4f}")

# Evaluation
svhn_model_frozen.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in svhn_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = svhn_model_frozen(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_frozen = 100 * correct / total
print(f"\nTest Accuracy (Frozen Layers): {accuracy_frozen:.2f}%\n")


--- Training Frozen Model on SVHN ---
Epoch 1/5, Loss: 2.2407
Epoch 2/5, Loss: 2.2278
Epoch 3/5, Loss: 2.2204
Epoch 4/5, Loss: 2.2138
Epoch 5/5, Loss: 2.2075

Test Accuracy (Frozen Layers): 20.08%



### 2.2. Strategy B: Fine-Tuning (All Layers)

Now, let's try fine-tuning. We'll load the pre-trained weights again, but this time we'll allow all the layers to be updated during training.

In [11]:
#  model instance for fine-tuning
svhn_model_finetune = CNN(num_classes=10)

# Copy weights, same as before
pretrained_dict = pretrained_model.state_dict()
model_dict = svhn_model_finetune.state_dict()
pretrained_dict = {k: v for k, v in pretrained_dict.items() if "fc2" not in k}
model_dict.update(pretrained_dict)
svhn_model_finetune.load_state_dict(model_dict)

# Ensure all parameters are trainable (this is the default behavior)
for param in svhn_model_finetune.parameters():
    param.requires_grad = True

print("Fine-Tuning Model Parameter Status:")
for name, param in svhn_model_finetune.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

svhn_model_finetune.to(device)

# Optimizer will update ALL model parameters
optimizer = torch.optim.Adam(svhn_model_finetune.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

Fine-Tuning Model Parameter Status:
conv1.weight: requires_grad=True
conv1.bias: requires_grad=True
conv2.weight: requires_grad=True
conv2.bias: requires_grad=True
fc1.weight: requires_grad=True
fc1.bias: requires_grad=True
fc2.weight: requires_grad=True
fc2.bias: requires_grad=True


In [12]:
# Training loop for the fine-tuned model
print("\n--- Training Fine-Tuned Model on SVHN ---")
num_epochs = 5
for epoch in range(num_epochs):
    svhn_model_finetune.train()
    running_loss = 0.0
    for images, labels in svhn_train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = svhn_model_finetune(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(svhn_train_loader):.4f}")

# Evaluation
svhn_model_finetune.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in svhn_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = svhn_model_finetune(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_finetune = 100 * correct / total
print(f"\nTest Accuracy (Fine-Tuned): {accuracy_finetune:.2f}%\n")


--- Training Fine-Tuned Model on SVHN ---
Epoch 1/5, Loss: 0.8310
Epoch 2/5, Loss: 0.4480
Epoch 3/5, Loss: 0.3731
Epoch 4/5, Loss: 0.3231
Epoch 5/5, Loss: 0.2857

Test Accuracy (Fine-Tuned): 87.93%



---

## 3. Experiment 2: Transfer Learning on CIFAR-100 Dataset

The **CIFAR-100** dataset is more challenging. It has 100 classes, while our model was pre-trained on only 10 classes. The images are visually similar to CIFAR-10, but the task is more complex. Here, we need to replace the final layer with a new one that has 100 outputs.

In [13]:
# Download and load the CIFAR-100 dataset
cifar100_train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
cifar100_train_loader = DataLoader(cifar100_train_dataset, batch_size=64, shuffle=True)

cifar100_test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
cifar100_test_loader = DataLoader(cifar100_test_dataset, batch_size=64, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


### 3.1. Strategy A: Feature Extraction (Frozen Layers)

In [14]:
# Create a new model for CIFAR-100, replacing the final layer
cifar100_model_frozen = CNN(num_classes=100)

# Copy weights from pre-trained model, excluding the final layer
pretrained_dict = pretrained_model.state_dict()
model_dict = cifar100_model_frozen.state_dict()
pretrained_dict = {k: v for k, v in pretrained_dict.items() if "fc2" not in k}
model_dict.update(pretrained_dict)
cifar100_model_frozen.load_state_dict(model_dict)

# Freeze all layers except the new fc2
for name, param in cifar100_model_frozen.named_parameters():
    if "fc2" not in name:
        param.requires_grad = False

cifar100_model_frozen.to(device)

# Optimizer will only update the parameters of fc2
optimizer = optim.Adam(filter(lambda p: p.requires_grad, cifar100_model_frozen.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [15]:
# Training loop for the frozen CIFAR-100 model
print("--- Training Frozen Model on CIFAR-100 ---")
num_epochs = 5
for epoch in range(num_epochs):
    cifar100_model_frozen.train()
    running_loss = 0.0
    for images, labels in cifar100_train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = cifar100_model_frozen(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(cifar100_train_loader):.4f}")

# Evaluation
cifar100_model_frozen.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in cifar100_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cifar100_model_frozen(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_cifar100_frozen = 100 * correct / total
print(f"\nTest Accuracy (Frozen Layers): {accuracy_cifar100_frozen:.2f}%\n")

--- Training Frozen Model on CIFAR-100 ---
Epoch 1/5, Loss: 4.5577
Epoch 2/5, Loss: 4.4692
Epoch 3/5, Loss: 4.3944
Epoch 4/5, Loss: 4.3302
Epoch 5/5, Loss: 4.2743

Test Accuracy (Frozen Layers): 10.75%



### 3.2. Strategy B: Fine-Tuning (All Layers)

In [16]:
# Create another model instance for fine-tuning on CIFAR-100
cifar100_model_finetune = CNN(num_classes=100)

# Copy weights from pre-trained model, excluding the final layer
pretrained_dict = pretrained_model.state_dict()
model_dict = cifar100_model_finetune.state_dict()
pretrained_dict = {k: v for k, v in pretrained_dict.items() if "fc2" not in k}
model_dict.update(pretrained_dict)
cifar100_model_finetune.load_state_dict(model_dict)

# Ensure all layers are trainable
for param in cifar100_model_finetune.parameters():
    param.requires_grad = True

cifar100_model_finetune.to(device)

# Optimizer will update ALL model parameters
optimizer = optim.Adam(cifar100_model_finetune.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [17]:
# Training loop for the fine-tuned CIFAR-100 model
print("--- Training Fine-Tuned Model on CIFAR-100 ---")
num_epochs = 5
for epoch in range(num_epochs):
    cifar100_model_finetune.train()
    running_loss = 0.0
    for images, labels in cifar100_train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = cifar100_model_finetune(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(cifar100_train_loader):.4f}")

# Evaluation
cifar100_model_finetune.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in cifar100_test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cifar100_model_finetune(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_cifar100_finetune = 100 * correct / total
print(f"\nTest Accuracy (Fine-Tuned): {accuracy_cifar100_finetune:.2f}%\n")

--- Training Fine-Tuned Model on CIFAR-100 ---
Epoch 1/5, Loss: 3.5529
Epoch 2/5, Loss: 2.8303
Epoch 3/5, Loss: 2.4992
Epoch 4/5, Loss: 2.2535
Epoch 5/5, Loss: 2.0564

Test Accuracy (Fine-Tuned): 39.21%

